In [9]:
# 1. In day13.ipynb, define the required fields for each file.

required_fields = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ],

    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "shipping_limit_date"
    ],

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "products": [
        "product_id",
        "product_category_name",
        "product_weight_g"
    ],

    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
}

print(required_fields)


{'orders': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp'], 'order_items': ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value', 'shipping_limit_date'], 'customers': ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'], 'products': ['product_id', 'product_category_name', 'product_weight_g'], 'sellers': ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']}


In [33]:
 # 2. Validate every row and collect errors

def validate_columns(actual_columns, required_columns):

    missing_columns = [
        column for column in required_columns
        if column not in actual_columns
    ]

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    print("Schema validation passed!")

    

In [7]:
def validate_rows(rows, required_columns):

    valid_rows = []
    rejected_rows = []

    for row_number, row in enumerate(rows, start=1):

        errors = []

        for column in required_columns:

            if row[column] is None or str(row[column]).strip() == "":
                errors.append(f"Empty value: {column}")

        if errors:
            rejected_rows.append({
                "row_number": row_number,
                "data": row,
                "errors": errors
            })
        else:
            valid_rows.append(row)

    return valid_rows, rejected_rows

In [31]:
import pandas as pd

orders_df = pd.read_csv("../orders_dataset.csv")

required_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
]

# Task 1: Schema validation
validate_columns(orders_df.columns, required_columns)

# Task 2: Row validation
orders = orders_df.to_dict(orient="records")

valid, rejected = validate_rows(orders, required_columns)

print("Total:", len(orders))
print("Valid:", len(valid))
print("Rejected:", len(rejected))

for item in rejected:
    print(item["row_number"], item["errors"])

Schema validation passed!
Total: 49
Valid: 49
Rejected: 0


In [34]:
# 3. Write rejected rows to bad_records.csv with a reason


import csv

def write_bad_records(rejected_rows, filename="bad_records.csv"):

    if not rejected_rows:
        print("No rejected records found.")
        return

    fieldnames = ["row_number"] + list(rejected_rows[0]["data"].keys()) + ["reason"]

    with open(filename, "w", newline="", encoding="utf-8") as file:

        writer = csv.DictWriter(file, fieldnames=fieldnames)

        writer.writeheader()

        for item in rejected_rows:

            record = {
                "row_number": item["row_number"],
                **item["data"],
                "reason": "; ".join(item["errors"])
            }

            writer.writerow(record)

    print(f"{len(rejected_rows)} rejected records saved to {filename}")
    
write_bad_records(rejected)

No rejected records found.


In [35]:
# 4. Check data types (numeric price, parseable dates).
# VALIDATION

from datetime import datetime

def validate_datatypes(rows):

    valid_rows = []
    rejected_rows = []

    for row_number, row in enumerate(rows, start=1):

        errors = []

        # Check price
        try:
            float(row["price"])
        except (ValueError, TypeError):
            errors.append("Invalid price: must be numeric")

        # Check freight value
        try:
            float(row["freight_value"])
        except (ValueError, TypeError):
            errors.append("Invalid freight_value: must be numeric")

        # Check shipping date
        try:
            datetime.strptime(
                row["shipping_limit_date"],
                "%Y-%m-%d %H:%M:%S"
            )
        except (ValueError, TypeError):
            errors.append("Invalid shipping_limit_date: incorrect date format")

        if errors:
            rejected_rows.append({
                "row_number": row_number,
                "data": row,
                "errors": errors
            })
        else:
            valid_rows.append(row)

    return valid_rows, rejected_rows


In [39]:
import pandas as pd

items_df = pd.read_csv(
    "../olist_order_items_dataset.csv",
    dtype=str
)

items = items_df.to_dict(orient="records")

valid_items, rejected_items = validate_datatypes(items)

print("Total records:", len(items))
print("Valid records:", len(valid_items))
print("Rejected records:", len(rejected_items))

Total records: 112650
Valid records: 112649
Rejected records: 1


In [40]:
# 5. Business rules: price > 0, order_status in the allowed list.

allowed_status = [
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "unavailable",
    "canceled"
]


def validate_business_rules(rows, dataset):

    valid_rows = []
    rejected_rows = []

    for row_number, row in enumerate(rows, start=1):

        errors = []

        # Rule 1: Price must be greater than zero
        if dataset == "order_items":

            try:
                if float(row["price"]) <= 0:
                    errors.append("Price must be greater than zero")
            except (ValueError, TypeError, KeyError):
                errors.append("Invalid price for business rule check")

        # Rule 2: Order status must be from allowed list
        elif dataset == "orders":

            if row["order_status"] not in allowed_status:
                errors.append("Invalid order status")

        if errors:
            rejected_rows.append({
                "row_number": row_number,
                "data": row,
                "errors": errors
            })
        else:
            valid_rows.append(row)

    return valid_rows, rejected_rows


In [42]:
import pandas as pd

items_df = pd.read_csv(
    "../olist_order_items_dataset.csv",
    dtype=str
)

items = items_df.to_dict(orient="records")

valid_prices, rejected_prices = validate_business_rules(
    items, "order_items"
)

print("Total:", len(items))
print("Valid:", len(valid_prices))
print("Rejected:", len(rejected_prices))

Total: 112650
Valid: 112650
Rejected: 0


In [43]:

# 6. Regex validation

import re

# Define regex rules for each dataset
regex_rules = {
    "orders": {
        "order_id": r"[0-9a-fA-F]{32}",
        "customer_id": r"[0-9a-fA-F]{32}"
    },

    "order_items": {
        "order_id": r"[0-9a-fA-F]{32}",
        "product_id": r"[0-9a-fA-F]{32}",
        "seller_id": r"[0-9a-fA-F]{32}"
    },

    "customers": {
        "customer_id": r"[0-9a-fA-F]{32}",
        "customer_unique_id": r"[0-9a-fA-F]{32}",
        "customer_zip_code_prefix": r"[0-9]{5}"
    },

    "sellers": {
        "seller_id": r"[0-9a-fA-F]{32}",
        "seller_zip_code_prefix": r"[0-9]{5}"
    },

    "geolocation": {
        "geolocation_zip_code_prefix": r"[0-9]{5}"
    }
}


def validate_regex(rows, dataset):

    valid_rows = []
    rejected_rows = []

    rules = regex_rules[dataset]

    for row_number, row in enumerate(rows, start=1):

        errors = []

        for column, pattern in rules.items():

            value = row.get(column, "")

            if not re.fullmatch(pattern, str(value)):
                errors.append(f"Invalid format: {column}")

        if errors:
            rejected_rows.append({
                "row_number": row_number,
                "data": row,
                "errors": errors
            })
        else:
            valid_rows.append(row)

    return valid_rows, rejected_rows


In [49]:
import pandas as pd

customers_df = pd.read_csv(
    "../customers.csv",
    dtype=str,
    keep_default_na=False
)

customers = customers_df.to_dict(orient="records")
valid_customers, rejected_customers = validate_regex(
    customers,
    "customers"
)

print("Total records:", len(customers))
print("Valid records:", len(valid_customers))
print("Rejected records:", len(rejected_customers))
for n in rejected_customers[:10]:
    print("Row:", n["row_number"])
    print("Errors:", n["errors"])

Total records: 39
Valid records: 38
Rejected records: 1
Row: 13
Errors: ['Invalid format: customer_id']


In [2]:

# 7. Create custom exceptions: SchemaError and FileFormatError.

def check_schema(actual_columns, required_columns):

    missing_columns = [
        column for column in required_columns
        if column not in actual_columns
    ]

    if missing_columns:
        class SchemaError(Exception):
            pass


        class FileFormatError(Exception):
            pass


        def check_schema(actual_columns, required_columns):
            missing_columns = [
                column for column in required_columns
                if column not in actual_columns
            ]

            if missing_columns:
                raise SchemaError(f"Missing required columns: {missing_columns}")

            print("Schema validation passed!")

    print("Schema validation passed!")


In [3]:
import pandas as pd

orders_df = pd.read_csv("../orders.csv")

required_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
]

try:
    check_schema(orders_df.columns, required_columns)

except SchemaError as e:
    print("Schema Error:", e)

Schema validation passed!


In [5]:

# Task 8: Handle missing and empty files

import os
import pandas as pd


def read_file(filename):

    try:
        if not os.path.exists(filename):
            raise FileNotFoundError(f"File not found: {filename}")

        if os.path.getsize(filename) == 0:
            raise FileFormatError("The CSV file is empty")

        df = pd.read_csv(filename, dtype=str, keep_default_na=False)

        if df.empty:
            raise FileFormatError("The CSV contains headers but no records")

        return df

    except FileNotFoundError as e:
        print("Missing File Error:", e)

    except pd.errors.EmptyDataError:
        print("File Format Error: No columns found in CSV")

    except pd.errors.ParserError as e:
        print("File Format Error:", e)

    except FileFormatError as e:
        print("File Format Error:", e)

    return None
orders_df = read_file("../orders.csv")

if orders_df is not None:
    print("File loaded successfully!")
    print("Total records:", len(orders_df))
    print(orders_df.head())


File loaded successfully!
Total records: 49
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  47770eb9100c2d1c44946d9cf07ec65e  51ce2a54b0b03bf3443c3d931a367089   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1   

In [11]:

# Task 9: Validation Summary

def validation_summary(total_rows, valid_rows, rejected_rows):

    total = total_rows
    valid = len(valid_rows)
    rejected = len(rejected_rows)

    rejection_rate = (rejected / total * 100) if total else 0
    valid_rate = (valid / total * 100) if total else 0

    print("=" * 45)
    print("          DATA VALIDATION REPORT")
    print("=" * 45)

    print(f"Total Records    : {total}")
    print(f"Valid Records    : {valid}")
    print(f"Rejected Records : {rejected}")

    print("-" * 45)

    print(f"Valid Rate       : {valid_rate:.2f}%")
    print(f"Rejection Rate   : {rejection_rate:.2f}%")

    print("=" * 45)

    return {
        "total": total,
        "valid": valid,
        "rejected": rejected,
        "valid_rate": valid_rate,
        "rejection_rate": rejection_rate
    }


In [4]:
orders_df = read_file("../orders.csv")
if orders_df is not None:
    orders = orders_df.to_dict(orient="records")
    valid_rows, rejected_rows = validate_rows(
        orders,
        required_fields["orders"]
    )

    summary = validation_summary(
        len(orders),
        valid_rows,
        rejected_rows
    )

else:
    print("Validation skipped because the dataset could not be loaded.")

NameError: name 'read_file' is not defined

In [1]:
def check_rejection_threshold(summary, max_rejection_percent=20.0):

    rejection_rate = summary["rejection_rate"]

    if summary["total"] == 0:
        raise FileFormatError("Cannot validate an empty dataset.")

    if rejection_rate > max_rejection_percent:
        raise ValueError(
            f"File rejected! Rejection rate is "
            f"{rejection_rate:.2f}%, which exceeds the allowed "
            f"limit of {max_rejection_percent:.2f}%."
        )

    print(
        f"File accepted! Rejection rate: {rejection_rate:.2f}% "
        f"(Allowed: {max_rejection_percent:.2f}%)."
    )

    return True

In [12]:
import csv
orders_df = read_file("../orders.csv")

if orders_df is not None:
    orders = orders_df.to_dict(orient="records")
    valid_rows, rejected_rows = validate_rows(
        orders,
        required_fields["orders"]
    )
    summary = validation_summary(
        len(orders),
        valid_rows,
        rejected_rows
    )
    try:
        check_rejection_threshold(
            summary,
            max_rejection_percent=20.0
        )
    except ValueError as e:
        print(e)
else:
    print("File validation could not be completed.")

          DATA VALIDATION REPORT
Total Records    : 49
Valid Records    : 49
Rejected Records : 0
---------------------------------------------
Valid Rate       : 100.00%
Rejection Rate   : 0.00%
File accepted! Rejection rate: 0.00% (Allowed: 20.00%).
